# 📊 Resumo de Dados - IBAMA Embargos

## Objetivo
Este notebook automatiza a análise dos arquivos Parquet/GeoParquet do IBAMA (camada Bronze), consolidando informações sobre:
- Tipos de dados e schema
- Contagem de registros e valores nulos (%)
- Estatísticas descritivas (min, max, média, desvio padrão, mediana)
- Valores únicos e amostras
- Uso de memória por arquivo

## Saída
Gera um arquivo `resumo_detalhado_ibama.parquet` com metadados consolidados para futura análise de qualidade dos dados.

In [1]:
import os
import pandas as pd
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm
import numpy as np
from datetime import datetime

# Configuração de caminhos
BASE_DIR = Path("/home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios")
IBAMA_BRONZE_DIR = BASE_DIR / "ibama/data/bronze/ibama_embargos"
OUTPUT_DIR = BASE_DIR / "ibama/data/resume"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def convert_to_numpy_dtypes(df):
    """Converte colunas com dtype nullable do pandas para numpy dtype."""
    for col in df.columns:
        dtype = df[col].dtype
        # Converte StringDtype para object
        if isinstance(dtype, pd.StringDtype):
            df[col] = df[col].astype('object')
        # Converte dtypes nullable Int64, Float64 etc para numpy
        elif isinstance(dtype, pd.Int64Dtype):
            df[col] = df[col].astype('float64')
        elif isinstance(dtype, pd.Float64Dtype):
            df[col] = df[col].astype('float64')
    return df

def get_numeric_stats(series):
    """Calcula estatísticas descritivas para colunas numéricas."""
    stats = {
        "min": None,
        "max": None,
        "mean": None,
        "std": None,
        "median": None
    }
    
    # Verifica se é numérico
    if not pd.api.types.is_numeric_dtype(series):
        return stats
    
    try:
        min_val = series.min()
        max_val = series.max()
        
        if pd.notna(min_val):
            stats["min"] = float(min_val)
        if pd.notna(max_val):
            stats["max"] = float(max_val)
            
        mean_val = series.mean()
        if pd.notna(mean_val):
            stats["mean"] = float(mean_val)
            
        std_val = series.std()
        if pd.notna(std_val):
            stats["std"] = float(std_val)
            
        median_val = series.median()
        if pd.notna(median_val):
            stats["median"] = float(median_val)
    except Exception:
        pass
    
    return stats

def analyze_file(file_path):
    """Analisa um arquivo parquet/geoparquet e retorna resumo detalhado por coluna."""
    name = file_path.name
    print(f"Analisando arquivo: {name}")
    
    try:
        # Leitura do arquivo (prioriza geopandas para geoparquet)
        is_geoparquet = file_path.suffix.lower() == '.geoparquet'
        
        if is_geoparquet:
            df = gpd.read_parquet(file_path)
        else:
            # Tenta pandas primeiro, fallback para geopandas
            try:
                df = pd.read_parquet(file_path)
            except Exception:
                df = gpd.read_parquet(file_path)
        
        # Converte dtypes nullable para numpy
        df = convert_to_numpy_dtypes(df)
            
        num_rows = len(df)
        
        if num_rows == 0:
            print(f"  ⚠️ Arquivo {name} está vazio.")
            return None

        # Memória em MB
        mem_usage = df.memory_usage(deep=True).sum() / (1024**2)
        
        summary = []
        for col in df.columns:
            series = df[col]
            null_count = int(series.isna().sum())
            dtype_str = str(series.dtype)
            
            stats = {
                "file": name,
                "total_rows": num_rows,
                "mem_mb": round(mem_usage, 2),
                "column": col,
                "dtype": dtype_str,
                "non_null_count": int(series.count()),
                "null_count": null_count,
                "null_percent": round((null_count / num_rows) * 100, 2),
                "unique_values": 0,
                "min": None,
                "max": None,
                "mean": None,
                "std": None,
                "median": None,
                "sample_values": "[]"
            }

            # Cálculo seguro de valores únicos (evita erro em tipos não hashable)
            try:
                stats["unique_values"] = int(series.nunique())
                sample = series.dropna().unique()[:3].tolist()
                stats["sample_values"] = str(sample)
            except Exception:
                stats["sample_values"] = "Erro ao extrair amostra"

            # Min/Max Seguro: Pula colunas de geometria
            is_geometry = "geometry" in dtype_str.lower() or "geo" in dtype_str.lower()
            if not is_geometry:
                numeric_stats = get_numeric_stats(series)
                stats.update(numeric_stats)
                
            summary.append(stats)
            
        print(f"  ✅ {num_rows} linhas, {len(df.columns)} colunas, {round(mem_usage, 2)} MB")
        return summary
    
    except Exception as e:
        print(f"  ❌ Erro crítico ao analisar {name}: {str(e)}")
        return None

# Busca arquivos parquet/geoparquet (case-insensitive)
print(f"📅 Data da análise: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 Diretório de entrada: {IBAMA_BRONZE_DIR}")
print(f"📁 Diretório de saída: {OUTPUT_DIR}\n")

files_to_analyze = list(IBAMA_BRONZE_DIR.glob("**/*.[pP][aA][rR][qQ][uU][eE][tT]")) + \
                   list(IBAMA_BRONZE_DIR.glob("**/*.[gG][eE][oO][pP][aA][rR][qQ][uU][eE][tT]"))

if not files_to_analyze:
    print("❌ Nenhum arquivo parquet/geoparquet encontrado.")
else:
    print(f"📄 Arquivos encontrados: {len(files_to_analyze)}\n")

all_summaries = []
for file_path in tqdm(files_to_analyze, desc="Progresso"):
    file_summary = analyze_file(file_path)
    if file_summary:
        all_summaries.extend(file_summary)

if all_summaries:
    summary_df = pd.DataFrame(all_summaries)
    
    # Salva em parquet
    output_file = OUTPUT_DIR / "resumo_detalhado_ibama.parquet"
    summary_df.to_parquet(output_file, index=False)
    
    print(f"\n{'='*60}")
    print(f"✅ Resumo exportado com sucesso!")
    print(f"   Arquivo: {output_file}")
    print(f"   Total de colunas analisadas: {len(summary_df)}")
    print(f"{'='*60}")
    
    # Exibe preview
    display(summary_df.head(10))
else:
    print("\n❌ Nenhum dado foi processado. Verifique se os caminhos estão corretos.")

📅 Data da análise: 2026-03-28 21:00:14
📁 Diretório de entrada: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/ibama/data/bronze/ibama_embargos
📁 Diretório de saída: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/ibama/data/resume

📄 Arquivos encontrados: 2



Progresso:   0%|                                                                                                      | 0/2 [00:00<?, ?it/s]

Analisando arquivo: embargos_ibama_tabular.parquet


Progresso:  50%|███████████████████████████████████████████████                                               | 1/2 [00:01<00:01,  1.09s/it]

  ✅ 88586 linhas, 38 colunas, 170.2 MB
Analisando arquivo: embargos_ibama_full.geoparquet


Progresso: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.68s/it]

Progresso: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.59s/it]

  ✅ 88586 linhas, 39 colunas, 170.88 MB

✅ Resumo exportado com sucesso!
   Arquivo: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/ibama/data/resume/resumo_detalhado_ibama.parquet
   Total de colunas analisadas: 77


,file,total_rows,mem_mb,column,dtype,non_null_count,null_count,null_percent,unique_values,min,max,mean,std,median,sample_values
0,embargos_ibama_tabular.parquet,88586,170.2,objectid,int64,88586,0,0.00,88586,1.0,88586.0,4.429350e+04,2.557272e+04,44293.5,"[25298, 25299, 25300]"
1,embargos_ibama_tabular.parquet,88586,170.2,seq_tad,int64,88586,0,0.00,87533,0.0,1876749.0,1.231892e+06,6.043362e+05,1491242.0,"[1639007, 1472789, 1610984]"
2,embargos_ibama_tabular.parquet,88586,170.2,num_tad,object,88586,0,0.00,87698,NaN,NaN,NaN,NaN,NaN,"['756611', '602347', '729878']"
3,embargos_ibama_tabular.parquet,88586,170.2,serie_tad,object,63971,24615,27.79,5,NaN,NaN,NaN,NaN,NaN,"['E', 'C', 'A']"
4,embargos_ibama_tabular.parquet,88586,170.2,operacao,object,9465,79121,89.32,747,NaN,NaN,NaN,NaN,NaN,"['ONDA VERDE P11', 'ONDA VERDE', 'CONTROLE REM..."
5,embargos_ibama_tabular.parquet,88586,170.2,origem_geo,object,88586,0,0.00,3,NaN,NaN,NaN,NaN,NaN,"['Polígono', 'Sem Geometria', 'Ponto']"
6,embargos_ibama_tabular.parquet,88586,170.2,cod_uf,object,88586,0,0.00,27,NaN,NaN,NaN,NaN,NaN,"['13', '29', '15']"
7,embargos_ibama_tabular.parquet,88586,170.2,uf,object,88586,0,0.00,27,NaN,NaN,NaN,NaN,NaN,"['AM', 'BA', 'PA']"
8,embargos_ibama_tabular.parquet,88586,170.2,cod_munici,int64,88586,0,0.00,3769,1100015.0,9999999.0,2.485295e+06,1.391935e+06,1716208.0,"[1302405, 2927705, 1500602]"
9,embargos_ibama_tabular.parquet,88586,170.2,municipio,object,88586,0,0.00,3633,NaN,NaN,NaN,NaN,NaN,"['Lábrea', 'Santa Cruz Cabrália', 'Altamira']"
